# ConvTran: Pooling Experiments

We test whether the way ConvTran summarizes the sequence before the classifier changes its accuracy, training time or memory use. Both variants keep the original front-end, tAPE and eRPE, the Transformer block and the linear classifier. They differ only in the final pooling over time. For each run we record test accuracy, test loss, the time per training epoch and the peak CUDA memory during training and validation.

The ConvTran paper describes average and max pooling before the classifier, whereas the code at the pinned commit uses only global average pooling (`nn.AdaptiveAvgPool1d(1)`). The avg_max variant is an experimental variant motivated by this difference. It is not the paper's exact formulation and not part of the released code.

The `upstream/ConvTran` submodule is used at its pinned commit and is not modified. The new pooling is defined in `scripts/pooling_variants.py`.

| Variant | Pooling over time | Classifier input |
|---|---|---|
| average | Global average, as in the upstream code | 16 features |
| avg_max | avg + tanh(beta) × max, with beta trainable and initialized to zero | 16 features |

In the avg_max variant, the average and the maximum are taken over the same temporal axis of the Transformer output. Beta is a single trainable scalar shared by all features. Because tanh(0) = 0, the model starts as the original average pooling and can learn how much of the maximum to add, between −1 and 1 times. The variant adds one trainable parameter and keeps the dimension seen by the classifier.

Both variants share the upstream loader and its cached preprocessing, the fixed stratified validation split (`random_state=1234`), the RAdam optimizer and learning rate, and the evaluation on the official test split of the checkpoint with the lowest validation loss. For a given seed, both variants start from the same weights, and the random state used for shuffling and dropout is the same. Synthetic checks are in `tests/test_pooling_variants.py`.

## Configuration

The notebook has three modes. The pilot trains both variants on BasicMotions for two epochs with seed 1234 to check the pipeline and the measurements. The comparison trains both variants on Handwriting, FaceDetection and PEMS-SF for 100 epochs with the five baseline seeds. The benchmark applies the same protocol to the 28 UEA datasets that completed the baseline, as in `04_positional_encoding_experiments.ipynb`.

| Setting | Pilot | Comparison | Benchmark |
|---|---|---|---|
| Datasets | BasicMotions | Handwriting, FaceDetection, PEMS-SF | 28 baseline datasets |
| Seeds | 1234 | 1234, 2345, 3456, 4567, 5678 | 1234, 2345, 3456, 4567, 5678 |
| Epochs | 2 | 100 | 100 |
| Batch size | 16 | 16 | 16 |
| Embedding dimension | 16 | 16 | 16 |
| Attention heads | 8 | 8 | 8 |
| Runs | 2 | 30 | 280 |

The pilot and the comparison are complete, and `MODE` is set to the benchmark. The comparison runs have the same configuration as the benchmark runs on their three datasets, so the benchmark reuses them. Training starts only when `RUN_TRAINING` is `True`; otherwise Run All reads the saved results. This cell also checks the runner and the submodule commit.

In [1]:
from pathlib import Path
import math
import shlex
import subprocess
import sys

import pandas as pd

MODE = "benchmark"  # "pilot", "comparison" or "benchmark"
RUN_TRAINING = True  # Set to True, then Run All, to train the selected mode.

VARIANTS = ["average", "avg_max"]
EXPECTED_SEEDS = [1234, 2345, 3456, 4567, 5678]
BATCH_SIZE = 16
EMB_SIZE = 16
NUM_HEADS = 8
MODES = {
    "pilot": {"datasets": ["BasicMotions"], "seeds": [1234], "epochs": 2},
    "comparison": {
        "datasets": ["Handwriting", "FaceDetection", "PEMS-SF"],
        "seeds": EXPECTED_SEEDS,
        "epochs": 100,
    },
    # The 28 benchmark datasets are read from the baseline results in the run plan.
    "benchmark": {"datasets": None, "seeds": EXPECTED_SEEDS, "epochs": 100},
}
assert MODE in MODES, f"MODE must be one of {sorted(MODES)}"
SEEDS = MODES[MODE]["seeds"]
EPOCHS = MODES[MODE]["epochs"]

ROOT = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "upstream" / "ConvTran" / "main.py").is_file()
    ),
    None,
)
assert ROOT is not None, "Open this notebook from the project repository."

RUNNER = ROOT / "scripts" / "run_pooling_experiment.py"
UPSTREAM = ROOT / "upstream" / "ConvTran"
DATA_ROOT = ROOT / "Dataset" / "UEA" / "Multivariate_ts"
BASELINE_RUNS_PATH = ROOT / "results" / "baseline_runs.csv"
assert RUNNER.is_file(), f"Missing experiment runner: {RUNNER}"
assert BASELINE_RUNS_PATH.is_file(), f"Missing baseline results: {BASELINE_RUNS_PATH}"

pinned = subprocess.check_output(
    ["git", "-C", str(ROOT), "ls-tree", "HEAD", "upstream/ConvTran"], text=True
).split()[2]
checked_out = subprocess.check_output(
    ["git", "-C", str(UPSTREAM), "rev-parse", "HEAD"], text=True
).strip()
upstream_changes = subprocess.check_output(
    ["git", "-C", str(UPSTREAM), "status", "--porcelain"], text=True
).strip()
assert checked_out == pinned, f"Submodule at {checked_out}, pinned {pinned}"
assert not upstream_changes, "The upstream submodule has local changes."

sys.path.insert(0, str(ROOT / "scripts"))
from run_pooling_experiment import FIELDS, RESULTS_PATH, log_complete, run_directory
from pooling_variants import pooling_tag

def run_dir(dataset, variant, seed):
    return run_directory(
        variant, dataset, seed, EPOCHS, BATCH_SIZE, EMB_SIZE, NUM_HEADS
    )

def runner_args(dataset, variant, seed):
    return [
        "--variant", variant,
        "--dataset", dataset,
        "--seed", str(seed),
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--emb-size", str(EMB_SIZE),
        "--num-heads", str(NUM_HEADS),
    ]

def retry_command(dataset, variant, seed):
    python = Path(sys.executable)
    if python.is_relative_to(ROOT):
        python = python.relative_to(ROOT)
    return shlex.join([
        str(python), str(RUNNER.relative_to(ROOT)),
        *runner_args(dataset, variant, seed), "--retry",
    ])

def as_available(table):
    return table.astype(object).where(table.notna(), "not available")

print("Repository:", ROOT)
print("Python:", sys.executable)
print("Upstream commit (pinned, clean):", pinned)
print("Results CSV:", RESULTS_PATH.relative_to(ROOT))
print("Poolings:", ", ".join(f"{v} = {pooling_tag(v)}" for v in VARIANTS))
print(f"Mode: {MODE} ({EPOCHS} epochs, seeds {SEEDS})")
print("RUN_TRAINING:", RUN_TRAINING)

Repository: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction
Python: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/.venv/bin/python
Upstream commit (pinned, clean): 148afb6ca549915b7e78b05e2ec3b4ba6e052341
Results CSV: results/pooling_runs.csv
Poolings: average = avg, avg_max = avg-tanhbeta-max-beta0
Mode: benchmark (100 epochs, seeds [1234, 2345, 3456, 4567, 5678])
RUN_TRAINING: True


## Pooling check

This check builds both variants on the CPU from the same seed, with a synthetic input of 6 channels and 100 time steps. It does not use data or train.

It verifies three properties. With beta = 0, the avg_max pooling returns the same tensor as the original average pooling, both on a synthetic Transformer output and inside the full model. Beta receives a finite, non-zero gradient from a cross-entropy loss. Both variants give the classifier a tensor of shape (batch, 16).

In [2]:
import torch
from torch import nn
import pooling_variants

CHECK_CHANNELS, CHECK_LENGTH, CHECK_CLASSES, CHECK_BATCH = 6, 100, 4, 3
check_config = {
    "Net_Type": ["C-T"], "emb_size": EMB_SIZE, "num_heads": NUM_HEADS,
    "dim_ff": 256, "dropout": 0.01,
    "Data_shape": (CHECK_BATCH, CHECK_CHANNELS, CHECK_LENGTH),
    "num_labels": CHECK_CLASSES,
}

def build(variant):
    torch.manual_seed(0)
    return pooling_variants.model_factory(check_config, variant)

torch.manual_seed(1)
transformer_output = torch.randn(CHECK_BATCH, EMB_SIZE, CHECK_LENGTH)
series = torch.randn(CHECK_BATCH, CHECK_CHANNELS, CHECK_LENGTH)
labels = torch.arange(CHECK_BATCH) % CHECK_CLASSES

models = {variant: build(variant).eval() for variant in VARIANTS}
with torch.no_grad():
    pool_equal = torch.equal(
        pooling_variants.AvgMaxPool()(transformer_output),
        nn.AdaptiveAvgPool1d(1)(transformer_output),
    )
    features = {
        variant: pooling_variants.pooled_features(model, series)
        for variant, model in models.items()
    }
    logits = {variant: model(series) for variant, model in models.items()}

trainable = build("avg_max").train()
nn.functional.cross_entropy(trainable(series), labels).backward()
beta_gradient = trainable.gap.beta.grad

checks = pd.DataFrame([
    {"check": "beta = 0: pooling equals AdaptiveAvgPool1d(1)", "result": pool_equal},
    {"check": "beta = 0: classifier input equal in both variants",
     "result": torch.equal(features["average"], features["avg_max"])},
    {"check": "beta = 0: model output equal in both variants",
     "result": torch.equal(logits["average"], logits["avg_max"])},
    {"check": "beta gradient is finite and non-zero",
     "result": bool(torch.isfinite(beta_gradient) and beta_gradient.abs() > 0)},
    {"check": "classifier input shape equal in both variants",
     "result": features["average"].shape == features["avg_max"].shape},
])
assert checks["result"].all(), checks
display(checks)

display(pd.DataFrame([
    {
        "variant": variant,
        "pooling": type(model.gap).__name__,
        "classifier_input": tuple(features[variant].shape),
        "trainable_parameters": pooling_variants.trainable_parameters(model),
    }
    for variant, model in models.items()
]))
print(f"Gradient of beta at beta = 0: {beta_gradient.item():.6f}")

/home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/.venv/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/.venv/lib/python3.10/site-packages/torch/nn/modules/conv.py:459: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at ../aten/src/ATen/native/Convolution.cpp:1003.)
  return F.conv2d(input, weight, bias, self.stride,


,check,result
0,beta = 0: pooling equals AdaptiveAvgPool1d(1),True
1,beta = 0: classifier input equal in both variants,True
2,beta = 0: model output equal in both variants,True
3,beta gradient is finite and non-zero,True
4,classifier input shape equal in both variants,True


,variant,pooling,classifier_input,trainable_parameters
0,average,AdaptiveAvgPool1d,"(3, 16)",17884
1,avg_max,AvgMaxPool,"(3, 16)",17885


Gradient of beta at beta = 0: 0.406716


## Datasets and run plan

The benchmark datasets come from `results/baseline_runs.csv`, with the same rule as in `04_positional_encoding_experiments.ipynb`. A dataset is eligible when each of the five baseline seeds has a valid result at 100 epochs and batch size 16. The cell stops if the file has duplicate rows for a dataset and seed, and checks that 28 datasets are selected.

MotorImagery and EigenWorms are not eligible because the baseline ran out of GPU memory at batch size 16. Their feasibility pilots are documented in `03_baseline_training.ipynb`.

The plan contains every combination of dataset, variant and seed in each mode. The cell checks that the pilot, comparison and benchmark plans contain 2, 30 and 280 unique runs, and that the dataset files exist.

In [3]:
baseline = pd.read_csv(BASELINE_RUNS_PATH)
baseline = baseline[(baseline["epochs"] == 100) & (baseline["batch_size"] == 16)]
duplicates = baseline[baseline.duplicated(["dataset", "seed"], keep=False)]
if not duplicates.empty:
    raise ValueError(f"Duplicate baseline rows:\n{duplicates}")

valid = baseline[
    baseline["accuracy"].between(0, 1)
    & baseline["test_loss"].map(lambda value: math.isfinite(float(value)))
]
BENCHMARK_DATASETS = [
    dataset
    for dataset, group in valid.groupby("dataset", sort=False)
    if sorted(group["seed"].astype(int)) == EXPECTED_SEEDS
]
assert len(BENCHMARK_DATASETS) == 28, f"Expected 28 datasets, found {len(BENCHMARK_DATASETS)}"
assert set(MODES["comparison"]["datasets"]) <= set(BENCHMARK_DATASETS)
MODES["benchmark"]["datasets"] = BENCHMARK_DATASETS
print(f"Benchmark datasets ({len(BENCHMARK_DATASETS)}):", ", ".join(BENCHMARK_DATASETS))
print("Separate feasibility pilots, not in the benchmark: MotorImagery, EigenWorms\n")

PLANS = {
    mode: [
        (dataset, variant, seed)
        for dataset in settings["datasets"]
        for seed in settings["seeds"]
        for variant in VARIANTS
    ]
    for mode, settings in MODES.items()
}
EXPECTED_RUNS = {"pilot": 2, "comparison": 30, "benchmark": 280}
for mode, mode_plan in PLANS.items():
    assert len(set(mode_plan)) == len(mode_plan) == EXPECTED_RUNS[mode], (mode, len(mode_plan))
    settings = MODES[mode]
    print(
        f"{mode}: {len(mode_plan)} runs ({len(settings['datasets'])} datasets x "
        f"{len(VARIANTS)} variants x {len(settings['seeds'])} seeds, "
        f"{settings['epochs']} epochs)"
    )

for dataset in sorted({d for settings in MODES.values() for d in settings["datasets"]}):
    for split in ("TRAIN", "TEST"):
        path = DATA_ROOT / dataset / f"{dataset}_{split}.ts"
        assert path.is_file(), f"Missing dataset file: {path}"

DATASETS = MODES[MODE]["datasets"]
plan = PLANS[MODE]
print(f"\nSelected: {MODE}, {len(plan)} runs")

Benchmark datasets (28): BasicMotions, Epilepsy, ArticularyWordRecognition, AtrialFibrillation, CharacterTrajectories, Cricket, DuckDuckGeese, ERing, EthanolConcentration, FaceDetection, FingerMovements, HandMovementDirection, Handwriting, Heartbeat, InsectWingbeat, JapaneseVowels, Libras, LSST, NATOPS, PenDigits, PEMS-SF, RacketSports, SelfRegulationSCP1, SelfRegulationSCP2, PhonemeSpectra, StandWalkJump, UWaveGestureLibrary, SpokenArabicDigits
Separate feasibility pilots, not in the benchmark: MotorImagery, EigenWorms

pilot: 2 runs (1 datasets x 2 variants x 1 seeds, 2 epochs)
comparison: 30 runs (3 datasets x 2 variants x 5 seeds, 100 epochs)
benchmark: 280 runs (28 datasets x 2 variants x 5 seeds, 100 epochs)

Selected: benchmark, 280 runs


## Training

When training is enabled, runs are executed one at a time, and each prints its progress, dataset, variant, seed and log path. Each run calls `scripts/run_pooling_experiment.py`, which starts the upstream `main.py` in a separate process with its own log. The full benchmark takes a long time, and the cell can be stopped and run again.

Logs, checkpoints and configurations are saved under `outputs/experiments/pooling/<variant>/<dataset>/seed<seed>_ep<epochs>_bs<batch>_emb<emb>_h<heads>_pool-<pooling>/`, where the pooling tag is `avg` or `avg-tanhbeta-max-beta0`. Completed runs are added to `results/pooling_runs.csv`, apart from the results of earlier notebooks. A run is reused only if its log for the same configuration is complete, so finished runs are not trained again. Pilot runs have 2 epochs and therefore their own directories, so the benchmark never reuses them.

If a run fails, the cell prints the error, the log path and the retry command, then moves on. The incomplete log is kept, and later attempts stop at that run until it is retried explicitly. The `--retry` command renames the old log to `run.failed-<UTC time>.log` before training.

In [4]:
attempts = []
if not RUN_TRAINING:
    print("Training disabled; set RUN_TRAINING = True in the configuration cell.")

for index, (dataset, variant, seed) in enumerate(plan if RUN_TRAINING else [], 1):
    log = run_dir(dataset, variant, seed) / "run.log"
    print(
        f"\n[{index}/{len(plan)}] {dataset}, {variant}, seed {seed}\n"
        f"  log: {log.relative_to(ROOT)}",
        flush=True,
    )
    process = subprocess.run(
        [sys.executable, str(RUNNER), *runner_args(dataset, variant, seed)],
        cwd=ROOT, capture_output=True, text=True, check=False,
    )
    if process.stdout.strip():
        print(process.stdout.strip())
    if process.returncode:
        print(f"  FAILED with exit code {process.returncode}")
        print(process.stderr.strip())
        print(f"  Log: {log}")
        print(f"  Retry: {retry_command(dataset, variant, seed)}")
    attempts.append({
        "dataset": dataset,
        "variant": variant,
        "seed": seed,
        "exit_code": process.returncode,
    })

if attempts:
    attempts = pd.DataFrame(attempts)
    print(f"\nAttempted {len(attempts)} runs; {int((attempts['exit_code'] != 0).sum())} failed.")
    display(attempts[attempts["exit_code"] != 0])


[1/280] BasicMotions, average, seed 1234
  log: outputs/experiments/pooling/average/basicmotions/seed1234_ep100_bs16_emb16_h8_pool-avg/run.log
Training average on BasicMotions. Log: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/experiments/pooling/average/basicmotions/seed1234_ep100_bs16_emb16_h8_pool-avg/run.log
Saved result to: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/results/pooling_runs.csv

[2/280] BasicMotions, avg_max, seed 1234
  log: outputs/experiments/pooling/avg_max/basicmotions/seed1234_ep100_bs16_emb16_h8_pool-avg-tanhbeta-max-beta0/run.log
Training avg_max on BasicMotions. Log: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/experiments/pooling/avg_max/basicmotions/seed1234_ep100_bs16_emb16_h8_pool-avg-tanhbeta-max-beta0/run.log
Saved result to: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/results/pooling_runs.csv

[3/280] BasicMotions, ave

,dataset,variant,seed,exit_code


## Results by run

Results are read from `results/pooling_runs.csv` and the run directories, so this cell also works with training disabled. Only rows matching the selected poolings, epochs, batch size, embedding dimension, number of heads, datasets and seeds are used.

A run is complete when the CSV has a row for its exact configuration and its log contains the final test summary, the measurement record and the record of the selected checkpoint. A run is incomplete when its log exists but is not complete; it needs the `--retry` command. All other runs are pending.

Test accuracy and test loss come from the upstream evaluation of the checkpoint with the lowest validation loss on the official test split. For avg_max, beta and tanh(beta) are read from that checkpoint, together with its epoch. Epoch times and peak memory are measured as in `04_positional_encoding_experiments.ipynb`. The later-epoch time is the mean over epochs 2 to the last, which excludes the one-time costs of the first epoch. Peak memory is `torch.cuda.max_memory_allocated` over training and validation, and it is not available for CPU runs.

With more than 30 runs, the table shows the status counts per dataset instead of every run.

In [5]:
results = (
    pd.read_csv(RESULTS_PATH) if RESULTS_PATH.is_file()
    else pd.DataFrame(columns=FIELDS)
)
config_fields = ["dataset", "variant", "seed"]
selected = results[
    (results["epochs"] == EPOCHS)
    & (results["batch_size"] == BATCH_SIZE)
    & (results["emb_size"] == EMB_SIZE)
    & (results["num_heads"] == NUM_HEADS)
    & results["dataset"].isin(DATASETS)
    & results["variant"].isin(VARIANTS)
    & results["seed"].isin(SEEDS)
]
selected = selected[selected["pooling"] == selected["variant"].map(pooling_tag)]
if selected.duplicated(config_fields).any():
    raise ValueError("Duplicate rows for one configuration in the results CSV.")
recorded = {
    (row.dataset, row.variant, int(row.seed)): row
    for row in selected.itertuples(index=False)
}

def epoch_times(value):
    if pd.isna(value):
        return []
    return [float(part) for part in str(value).split(";") if part]

rows = []
for dataset, variant, seed in plan:
    log = run_dir(dataset, variant, seed) / "run.log"
    record = recorded.get((dataset, variant, seed))
    row = {
        "dataset": dataset, "variant": variant, "seed": seed,
        "log": str(log.relative_to(ROOT)),
    }
    if record is not None and log_complete(log):
        times = epoch_times(record.train_epoch_times_s)
        row.update(
            status="complete",
            test_accuracy_pct=record.accuracy * 100,
            test_loss=record.test_loss,
            later_epochs_mean_s=(
                sum(times[1:]) / len(times[1:]) if len(times) > 1 else None
            ),
            timed_epochs=len(times),
            peak_cuda_alloc_train_val_mib=record.peak_cuda_allocated_train_val_mib,
            best_epoch=record.best_epoch,
            beta=record.beta,
            tanh_beta=record.tanh_beta,
            device=record.device,
        )
    elif log.is_file():
        row["status"] = "incomplete"
    else:
        row["status"] = "pending"
    rows.append(row)

RUN_COLUMNS = [
    "dataset", "variant", "seed", "status", "test_accuracy_pct", "test_loss",
    "later_epochs_mean_s", "timed_epochs", "peak_cuda_alloc_train_val_mib",
    "best_epoch", "beta", "tanh_beta", "device", "log",
]
NUMERIC_COLUMNS = [
    "test_accuracy_pct", "test_loss", "later_epochs_mean_s",
    "peak_cuda_alloc_train_val_mib", "beta", "tanh_beta",
]
runs_table = pd.DataFrame(rows).reindex(columns=RUN_COLUMNS)
runs_table[NUMERIC_COLUMNS] = runs_table[NUMERIC_COLUMNS].apply(pd.to_numeric)
for column in ("timed_epochs", "best_epoch"):
    runs_table[column] = pd.to_numeric(runs_table[column]).astype("Int64")

counts = runs_table["status"].value_counts()
print(f"{MODE}: {len(runs_table)} planned")
for status in ("complete", "pending", "incomplete"):
    print(f"  {status}: {counts.get(status, 0)}")
if len(runs_table) <= 30:
    display(as_available(runs_table.round(4)))
else:
    progress = (
        runs_table.pivot_table(
            index="dataset", columns="status", values="seed",
            aggfunc="count", fill_value=0,
        )
        .reindex(index=DATASETS, columns=["complete", "pending", "incomplete"], fill_value=0)
        .rename_axis(columns=None)
    )
    with pd.option_context("display.max_rows", None):
        display(progress)

incomplete = runs_table[runs_table["status"] == "incomplete"]
if not incomplete.empty:
    print("Incomplete logs are kept. Inspect each log, then retry from the repository root:")
    for run in incomplete.itertuples(index=False):
        print(" ", retry_command(run.dataset, run.variant, run.seed))

benchmark: 280 planned
  complete: 280
  pending: 0
  incomplete: 0


,complete,pending,incomplete
dataset,,,
BasicMotions,10,0,0
Epilepsy,10,0,0
ArticularyWordRecognition,10,0,0
AtrialFibrillation,10,0,0
CharacterTrajectories,10,0,0
Cricket,10,0,0
DuckDuckGeese,10,0,0
ERing,10,0,0
EthanolConcentration,10,0,0


## Summary by dataset and variant

Completed seeds are summarized for each dataset and variant. Accuracy is the mean in percent with the sample standard deviation in percentage points. The standard deviation is not available with a single seed.

The epoch time excludes the first epoch: it is averaged over epochs 2 to the last within each seed, then across seeds. Peak memory and, for avg_max, the final tanh(beta) are averaged across seeds.

Summaries with pending seeds are labelled as partial and should not be compared directly with complete ones.

In [6]:
summary_rows = []
for dataset in DATASETS:
    for variant in VARIANTS:
        done = runs_table[
            (runs_table["dataset"] == dataset)
            & (runs_table["variant"] == variant)
            & (runs_table["status"] == "complete")
        ]
        completed = len(done)
        summary_rows.append({
            "dataset": dataset,
            "variant": variant,
            "summary": "complete" if completed == len(SEEDS) else "partial",
            "seeds_completed": f"{completed}/{len(SEEDS)}",
            "mean_accuracy_pct": done["test_accuracy_pct"].mean(),
            "std_accuracy_pp": (
                done["test_accuracy_pct"].std(ddof=1) if completed > 1 else None
            ),
            "mean_later_epochs_s": done["later_epochs_mean_s"].mean(),
            "mean_peak_cuda_alloc_train_val_mib": (
                done["peak_cuda_alloc_train_val_mib"].mean()
            ),
            "mean_final_tanh_beta": done["tanh_beta"].mean(),
        })

summary = pd.DataFrame(summary_rows)
SUMMARY_NUMERIC = summary.columns[4:]
summary[SUMMARY_NUMERIC] = summary[SUMMARY_NUMERIC].apply(pd.to_numeric)
with pd.option_context("display.max_rows", None):
    display(as_available(summary.round(4)))

,dataset,variant,summary,seeds_completed,mean_accuracy_pct,std_accuracy_pp,mean_later_epochs_s,mean_peak_cuda_alloc_train_val_mib,mean_final_tanh_beta
0,BasicMotions,average,complete,5/5,100.0,0.0,0.005,44.0703,not available
1,BasicMotions,avg_max,complete,5/5,100.0,0.0,0.0051,44.0723,0.0419
2,Epilepsy,average,complete,5/5,96.3768,0.0,0.0174,109.0815,not available
3,Epilepsy,avg_max,complete,5/5,96.3768,0.0,0.0182,109.0835,0.1525
4,ArticularyWordRecognition,average,complete,5/5,96.7333,0.7958,0.0311,73.5298,not available
5,ArticularyWordRecognition,avg_max,complete,5/5,96.9333,0.2789,0.0329,73.5317,0.4377
6,AtrialFibrillation,average,complete,5/5,32.0,2.9814,0.0103,635.104,not available
7,AtrialFibrillation,avg_max,complete,5/5,32.0,2.9814,0.0104,635.106,-0.0015
8,CharacterTrajectories,average,complete,5/5,99.0947,0.0853,0.1646,89.9038,not available
9,CharacterTrajectories,avg_max,complete,5/5,98.9833,0.1882,0.1725,89.9058,0.7941


## Paired accuracy difference

For each dataset and seed where both variants are complete, we take the avg_max accuracy minus the average accuracy, in percentage points. A positive value favours avg_max. Both runs of a pair share the seed, the validation split and the initial weights.

The table reports the number of pairs, the mean difference and its sample standard deviation. Datasets with fewer than five pairs are labelled as partial. The per-seed differences are shown below the summary when there are at most 30 pairs.

In [7]:
complete_runs = runs_table[runs_table["status"] == "complete"]
paired = (
    complete_runs.pivot_table(
        index=["dataset", "seed"], columns="variant", values="test_accuracy_pct"
    )
    .reindex(columns=VARIANTS)
    .dropna()
    .reindex(index=DATASETS, level="dataset")
    .rename_axis(columns=None)
    .reset_index()
)
paired["difference_pp"] = paired["avg_max"] - paired["average"]

difference_rows = []
for dataset in DATASETS:
    pairs = paired.loc[paired["dataset"] == dataset, "difference_pp"]
    difference_rows.append({
        "dataset": dataset,
        "summary": "complete" if len(pairs) == len(SEEDS) else "partial",
        "pairs": f"{len(pairs)}/{len(SEEDS)}",
        "mean_difference_pp": pairs.mean(),
        "std_difference_pp": pairs.std(ddof=1) if len(pairs) > 1 else None,
    })
differences = pd.DataFrame(difference_rows)
differences[["mean_difference_pp", "std_difference_pp"]] = (
    differences[["mean_difference_pp", "std_difference_pp"]].apply(pd.to_numeric)
)
print(f"Seeds with both variants complete: {len(paired)} of {len(DATASETS) * len(SEEDS)}")
with pd.option_context("display.max_rows", None):
    display(as_available(differences.round(4)))
if 0 < len(paired) <= 30:
    display(paired.round(4))

Seeds with both variants complete: 140 of 140


,dataset,summary,pairs,mean_difference_pp,std_difference_pp
0,BasicMotions,complete,5/5,0.0,0.0
1,Epilepsy,complete,5/5,0.0,0.0
2,ArticularyWordRecognition,complete,5/5,0.2,0.8692
3,AtrialFibrillation,complete,5/5,0.0,0.0
4,CharacterTrajectories,complete,5/5,-0.1114,0.2679
5,Cricket,complete,5/5,-0.2778,0.6211
6,DuckDuckGeese,complete,5/5,1.6,3.2863
7,ERing,complete,5/5,-0.3704,0.7407
8,EthanolConcentration,complete,5/5,0.076,2.4257
9,FaceDetection,complete,5/5,0.2724,0.7325


## Summary across datasets

The summary across datasets uses only datasets where both variants have all seeds of the mode complete. Each of these datasets has the same weight, whatever its size or difficulty. The cell reports how many datasets entered, and the summary is labelled as partial while datasets of the mode are missing.

For each variant we report the mean of the per-dataset mean accuracies, later-epoch times and peak memory, and for avg_max the mean final tanh(beta). The mean difference is the mean of the per-dataset paired differences. We also count the datasets where avg_max improves, ties or worsens, from the sign of its paired mean difference.

In [8]:
complete_datasets = [
    dataset for dataset in DATASETS
    if (summary.loc[summary["dataset"] == dataset, "summary"] == "complete").all()
]
label = "complete" if len(complete_datasets) == len(DATASETS) else "partial"
print(
    f"Summary across datasets ({label}): {len(complete_datasets)} of "
    f"{len(DATASETS)} datasets with every seed of the mode ({len(SEEDS)}) "
    "complete in both variants"
)

included = summary[summary["dataset"].isin(complete_datasets)]
overall = (
    included.groupby("variant")
    .agg(
        datasets=("dataset", "count"),
        mean_accuracy_pct=("mean_accuracy_pct", "mean"),
        mean_later_epochs_s=("mean_later_epochs_s", "mean"),
        mean_peak_cuda_alloc_train_val_mib=("mean_peak_cuda_alloc_train_val_mib", "mean"),
        mean_final_tanh_beta=("mean_final_tanh_beta", "mean"),
    )
    .reindex(VARIANTS)
    .reset_index()
)
overall["datasets"] = overall["datasets"].fillna(0).astype(int)
display(as_available(overall.round(4)))

dataset_differences = differences.set_index("dataset").loc[
    complete_datasets, "mean_difference_pp"
]
TIE_TOLERANCE_PP = 1e-9
outcomes = pd.DataFrame([{
    "summary": label,
    "datasets": len(complete_datasets),
    "mean_difference_pp": dataset_differences.mean(),
    "avg_max_better": int((dataset_differences > TIE_TOLERANCE_PP).sum()),
    "tie": int((dataset_differences.abs() <= TIE_TOLERANCE_PP).sum()),
    "avg_max_worse": int((dataset_differences < -TIE_TOLERANCE_PP).sum()),
}])
outcomes["mean_difference_pp"] = pd.to_numeric(outcomes["mean_difference_pp"])
display(as_available(outcomes.round(4)))

Summary across datasets (complete): 28 of 28 datasets with every seed of the mode (5) complete in both variants


,variant,datasets,mean_accuracy_pct,mean_later_epochs_s,mean_peak_cuda_alloc_train_val_mib,mean_final_tanh_beta
0,average,28,70.9612,0.5561,1188.3182,not available
1,avg_max,28,70.9715,0.5679,1188.321,0.263


,summary,datasets,mean_difference_pp,avg_max_better,tie,avg_max_worse
0,complete,28,0.0103,12,6,10


## Limitations

The five seeds of each dataset share one validation split, and each timing comes from a single run on a desktop GPU. With five pairs per dataset, each mean difference is a rough estimate. The benchmark gives every dataset the same weight and hides differences in size, length and number of classes. It excludes MotorImagery and EigenWorms, which did not fit in GPU memory under the baseline configuration.

The avg_max variant is one way to combine average and max pooling. It uses a single gate for all features and bounds the weight of the maximum between −1 and 1. Other combinations, such as concatenation with a wider classifier, are not tested. The final beta describes the selected checkpoint only and does not show how the weight changed during training.

The two-epoch pilot only checks that the pipeline and the measurements work. It is too short to compare accuracy.